# Smart Waste Collection Simulation
# Behaviour Analysis & Random Variable Preparation
#
# Objective:
# Analyse historical smart-bin behaviour and identify suitable
# random variables and probability distributions for the
# simulation model.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Load outputs
daily_df = pd.read_csv(
    "smart_bin_daily_transitions.csv",
    parse_dates=["timestamp", "previous_date"]
)

emptying_df = pd.read_csv(
    "smart_bin_emptying_events.csv",
    parse_dates=["timestamp", "previous_emptying_date"]
)

transition_prob = pd.read_csv(
    "smart_bin_transition_probabilities.csv"
)

print("Daily transitions:", daily_df.shape)
print("Potential emptying events:", emptying_df.shape)
print("Transition probability table:", transition_prob.shape)

print("\nDaily transition columns:")
print(daily_df.columns.tolist())

print("\nEmptying-event columns:")
print(emptying_df.columns.tolist())

print("\nTransition probability table:")
print(transition_prob)

Verify analysis population

In [ ]:
# ---------------------------------------------------------
# Step 2: Verify overall daily behaviour
# ---------------------------------------------------------

behaviour_counts = (
    daily_df["behaviour_type"]
    .value_counts()
)

behaviour_percent = (
    daily_df["behaviour_type"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

behaviour_summary = pd.DataFrame({
    "Count": behaviour_counts,
    "Percentage": behaviour_percent
})

print("Overall daily behaviour:")
print(behaviour_summary)

print("\nTotal transitions:")
print(behaviour_counts.sum())

print("\nFullness change values:")
print(sorted(daily_df["fullness_change"].unique()))

print("\nNumber of modelling bins:")
print(daily_df["serialNumber"].nunique())

In [ ]:
# ---------------------------------------------------------
# Step 3: Analyse positive daily fullness increases
# ---------------------------------------------------------

increase_df = daily_df[
    daily_df["fullness_change"] > 0
].copy()

increase_counts = (
    increase_df["fullness_change"]
    .value_counts()
    .sort_index()
)

increase_percent = (
    increase_df["fullness_change"]
    .value_counts(normalize=True)
    .sort_index()
    .mul(100)
    .round(2)
)

increase_summary = pd.DataFrame({
    "Count": increase_counts,
    "Percentage": increase_percent
})

print("Positive fullness increase distribution:")
print(increase_summary)

print("\nTotal positive increases:")
print(len(increase_df))

print("\nSummary statistics:")
print(increase_df["fullness_change"].describe())

In [ ]:
# ---------------------------------------------------------
# Step 4: Positive increase size by starting fullness
# ---------------------------------------------------------

increase_by_start_count = pd.crosstab(
    increase_df["previous_fullness"],
    increase_df["fullness_change"]
)

increase_by_start_pct = pd.crosstab(
    increase_df["previous_fullness"],
    increase_df["fullness_change"],
    normalize="index"
).mul(100).round(2)

print("Positive increase counts by starting fullness:")
print(increase_by_start_count)

print("\nPositive increase percentages by starting fullness:")
print(increase_by_start_pct)

print("\nNumber of positive increases from each starting state:")
print(
    increase_df["previous_fullness"]
    .value_counts()
    .sort_index()
)

In [ ]:
# ---------------------------------------------------------
# Step 5: Examine behaviour excluding potential emptying
# ---------------------------------------------------------

non_emptying_df = daily_df[
    daily_df["behaviour_type"] != "Potential Emptying"
].copy()

print("Transitions after excluding potential emptying:")
print(len(non_emptying_df))

print("\nBehaviour remaining:")
print(
    non_emptying_df["behaviour_type"]
    .value_counts()
)

print("\nBehaviour remaining (%):")
print(
    non_emptying_df["behaviour_type"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

# State transitions after removing potential emptying events
non_emptying_transition_pct = pd.crosstab(
    non_emptying_df["previous_fullness"],
    non_emptying_df["latestFullness"],
    normalize="index"
).mul(100).round(2)

print("\nState transitions excluding potential emptying (%):")
print(non_emptying_transition_pct)

In [ ]:
# ---------------------------------------------------------
# Step 6: Analyse accumulation-only behaviour
# Keep only No Change and Increase
# ---------------------------------------------------------

accumulation_df = daily_df[
    daily_df["behaviour_type"].isin(
        ["No Change", "Increase"]
    )
].copy()

print("Accumulation-only transitions:")
print(len(accumulation_df))

print("\nShare of all daily transitions:")
print(
    round(
        len(accumulation_df) / len(daily_df) * 100,
        2
    ),
    "%"
)

accumulation_count = pd.crosstab(
    accumulation_df["previous_fullness"],
    accumulation_df["latestFullness"]
)

accumulation_pct = pd.crosstab(
    accumulation_df["previous_fullness"],
    accumulation_df["latestFullness"],
    normalize="index"
).mul(100).round(2)

print("\nAccumulation transition counts:")
print(accumulation_count)

print("\nAccumulation transition probabilities (%):")
print(accumulation_pct)

print("\nObservations available for each starting fullness:")
print(
    accumulation_df["previous_fullness"]
    .value_counts()
    .sort_index()
)

In [ ]:
# ---------------------------------------------------------
# Step 7: Analyse potential emptying events by starting state
# ---------------------------------------------------------

potential_emptying_df = daily_df[
    daily_df["behaviour_type"] == "Potential Emptying"
].copy()

emptying_start_counts = (
    potential_emptying_df["previous_fullness"]
    .value_counts()
    .sort_index()
)

emptying_start_pct = (
    potential_emptying_df["previous_fullness"]
    .value_counts(normalize=True)
    .sort_index()
    .mul(100)
    .round(2)
)

emptying_start_summary = pd.DataFrame({
    "Count": emptying_start_counts,
    "Percentage_of_emptying_events": emptying_start_pct
})

print("Potential emptying events by starting fullness:")
print(emptying_start_summary)

print("\nTotal potential emptying events:")
print(len(potential_emptying_df))


# Probability of a potential emptying event from each starting state
state_totals = (
    daily_df["previous_fullness"]
    .value_counts()
    .sort_index()
)

emptying_probability_by_state = (
    emptying_start_counts / state_totals * 100
).fillna(0).round(2)

print("\nPotential emptying probability by starting fullness:")
print(emptying_probability_by_state)

In [ ]:
# ---------------------------------------------------------
# Step 8: Reliable time between potential emptying events
# ---------------------------------------------------------

reliable_intervals = emptying_df[
    (emptying_df["coverage_percent"] == 100) &
    (emptying_df["days_since_previous_emptying"].notna())
].copy()

intervals = reliable_intervals[
    "days_since_previous_emptying"
]

print("Reliable intervals:")
print(len(intervals))

print("\nSummary statistics:")
print(intervals.describe())

print("\nMost common interval lengths:")
print(
    intervals
    .value_counts()
    .sort_values(ascending=False)
    .head(15)
)

print("\nPercentiles:")
print(
    intervals.quantile(
        [0.50, 0.75, 0.90, 0.95, 0.99]
    )
)

In [ ]:
# ---------------------------------------------------------
# Step 9: Histogram of reliable potential-emptying intervals
# ---------------------------------------------------------

plt.figure(figsize=(10, 6))

plt.hist(
    intervals,
    bins=range(
        int(intervals.min()),
        int(intervals.max()) + 2
    ),
    edgecolor="black"
)

plt.title(
    "Distribution of Time Between Potential Emptying Events"
)

plt.xlabel(
    "Days Between Potential Emptying Events"
)

plt.ylabel(
    "Number of Observed Intervals"
)

plt.xlim(0, 40)

plt.grid(
    axis="y",
    alpha=0.3
)

plt.tight_layout()
plt.show()

In [ ]:
# ---------------------------------------------------------
# Step 10: Check mean and variance of reliable intervals
# ---------------------------------------------------------

interval_mean = intervals.mean()
interval_variance = intervals.var()
interval_std = intervals.std()

print("Mean:", round(interval_mean, 4))
print("Variance:", round(interval_variance, 4))
print("Standard deviation:", round(interval_std, 4))

print(
    "\nVariance / Mean ratio:",
    round(interval_variance / interval_mean, 4)
)

# Shift interval because minimum observed value is 2 days
shifted_intervals = intervals - 2

print("\nShifted interval summary:")
print(shifted_intervals.describe())

print("\nShifted mean:",
      round(shifted_intervals.mean(), 4))

print("Shifted variance:",
      round(shifted_intervals.var(), 4))

In [ ]:
# ---------------------------------------------------------
# Step 11: Estimate Negative Binomial parameters
# ---------------------------------------------------------

shift_mean = shifted_intervals.mean()
shift_var = shifted_intervals.var()

# Method-of-moments estimates
r = (shift_mean ** 2) / (shift_var - shift_mean)
p = r / (r + shift_mean)

print("Shifted mean:", round(shift_mean, 4))
print("Shifted variance:", round(shift_var, 4))

print("\nEstimated Negative Binomial parameters:")
print("r =", round(r, 4))
print("p =", round(p, 4))

# Check theoretical mean and variance from fitted parameters
nb_mean = r * (1 - p) / p
nb_variance = r * (1 - p) / (p ** 2)

print("\nModel check:")
print("Theoretical shifted mean:",
      round(nb_mean, 4))

print("Theoretical shifted variance:",
      round(nb_variance, 4))

In [ ]:
# ---------------------------------------------------------
# Step 12: Compare observed data with Negative Binomial fit
# ---------------------------------------------------------

from scipy.stats import nbinom

# Original interval lengths to display
days = np.arange(2, 31)

# Observed counts
observed_counts = (
    intervals
    .value_counts()
    .reindex(days, fill_value=0)
    .sort_index()
)

# Negative Binomial probabilities
# Shift by 2 because fitted distribution starts at 0
nb_probabilities = nbinom.pmf(
    days - 2,
    r,
    p
)

# Convert probabilities to expected counts
expected_counts = (
    nb_probabilities * len(intervals)
)

comparison_df = pd.DataFrame({
    "Days": days,
    "Observed": observed_counts.values,
    "Negative_Binomial": expected_counts
})

print(comparison_df.round(2))

# Plot
plt.figure(figsize=(10, 6))

plt.bar(
    comparison_df["Days"],
    comparison_df["Observed"],
    alpha=0.6,
    label="Observed"
)

plt.plot(
    comparison_df["Days"],
    comparison_df["Negative_Binomial"],
    marker="o",
    linewidth=2,
    label="Negative Binomial"
)

plt.title(
    "Observed vs Negative Binomial: "
    "Potential Emptying Intervals"
)

plt.xlabel(
    "Days Between Potential Emptying Events"
)

plt.ylabel(
    "Frequency"
)

plt.legend()
plt.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# ---------------------------------------------------------
# Step 13: Check behaviour variation between bins
# ---------------------------------------------------------

bin_summary = (
    daily_df
    .groupby("serialNumber")
    .agg(
        transitions=("behaviour_type", "size"),
        mean_change=("fullness_change", "mean"),
        increase_rate=(
            "behaviour_type",
            lambda x: (x == "Increase").mean() * 100
        ),
        no_change_rate=(
            "behaviour_type",
            lambda x: (x == "No Change").mean() * 100
        ),
        emptying_rate=(
            "behaviour_type",
            lambda x: (x == "Potential Emptying").mean() * 100
        )
    )
    .round(2)
)

print("Per-bin behaviour summary:")
print(bin_summary)

print("\nSummary across the 32 bins:")
print(
    bin_summary[
        [
            "increase_rate",
            "no_change_rate",
            "emptying_rate"
        ]
    ].describe().round(2)
)

In [ ]:
# ---------------------------------------------------------
# Step 14: Check behaviour by area and bin code
# ---------------------------------------------------------

bin_info = (
    daily_df[
        ["serialNumber", "description"]
    ]
    .drop_duplicates("serialNumber")
    .copy()
)

# Derive area from description
bin_info["area"] = np.where(
    bin_info["description"].str.contains(
        "Point Cook", case=False, na=False
    ),
    "Point Cook",
    "Werribee"
)

# Derive G / R code from description
bin_info["bin_code"] = (
    bin_info["description"]
    .str.extract(r"\b([GR])\s+(?:Werribee|Point Cook)\b")[0]
)

# Add area/code to per-bin summary
bin_summary2 = (
    bin_summary
    .reset_index()
    .merge(
        bin_info[
            ["serialNumber", "area", "bin_code"]
        ],
        on="serialNumber",
        how="left"
    )
)

print("Bin counts by area and code:")
print(
    pd.crosstab(
        bin_summary2["area"],
        bin_summary2["bin_code"]
    )
)

print("\nAverage behaviour by area:")
print(
    bin_summary2
    .groupby("area")[
        [
            "increase_rate",
            "no_change_rate",
            "emptying_rate"
        ]
    ]
    .mean()
    .round(2)
)

print("\nAverage behaviour by bin code:")
print(
    bin_summary2
    .groupby("bin_code")[
        [
            "increase_rate",
            "no_change_rate",
            "emptying_rate"
        ]
    ]
    .mean()
    .round(2)
)

In [ ]:
# ---------------------------------------------------------
# Step 15: Check descriptions not captured by G/R extraction
# ---------------------------------------------------------

unmatched_bins = bin_info[
    bin_info["bin_code"].isna()
]

print("Bins with no extracted G/R code:")
print(
    unmatched_bins[
        ["serialNumber", "description", "area"]
    ].to_string(index=False)
)

print("\nNumber of unmatched bins:")
print(len(unmatched_bins))

In [ ]:
# ---------------------------------------------------------
# Step 16: Time between positive fullness increases
# ---------------------------------------------------------

positive_events = daily_df[
    daily_df["behaviour_type"] == "Increase"
][
    ["serialNumber", "timestamp"]
].copy()

positive_events = positive_events.sort_values(
    ["serialNumber", "timestamp"]
)

positive_events["previous_increase_date"] = (
    positive_events
    .groupby("serialNumber")["timestamp"]
    .shift(1)
)

positive_events["days_between_increases"] = (
    positive_events["timestamp"]
    - positive_events["previous_increase_date"]
).dt.days

increase_intervals = (
    positive_events["days_between_increases"]
    .dropna()
)

print("Number of measurable intervals:")
print(len(increase_intervals))

print("\nSummary:")
print(increase_intervals.describe())

print("\nMost common intervals:")
print(
    increase_intervals
    .value_counts()
    .sort_values(ascending=False)
    .head(15)
)

print("\nPercentiles:")
print(
    increase_intervals.quantile(
        [0.50, 0.75, 0.90, 0.95, 0.99]
    )
)

In [ ]:
# ---------------------------------------------------------
# Step 17: Check coverage between positive increase events
# ---------------------------------------------------------

# Dates available for each bin in the modelling dataset
observed_dates = (
    daily_df[
        ["serialNumber", "timestamp"]
    ]
    .drop_duplicates()
)

coverage_results = []

for _, row in positive_events.dropna(
    subset=["previous_increase_date"]
).iterrows():

    bin_id = row["serialNumber"]
    start = row["previous_increase_date"]
    end = row["timestamp"]

    observed = observed_dates[
        (observed_dates["serialNumber"] == bin_id) &
        (observed_dates["timestamp"] > start) &
        (observed_dates["timestamp"] <= end)
    ]["timestamp"].nunique()

    expected = (end - start).days

    coverage_results.append(
        {
            "serialNumber": bin_id,
            "timestamp": end,
            "previous_increase_date": start,
            "days_between_increases": expected,
            "observed_days": observed,
            "expected_days": expected,
            "coverage_percent":
                observed / expected * 100
                if expected > 0 else np.nan
        }
    )

increase_coverage_df = pd.DataFrame(
    coverage_results
)

print("Coverage summary:")
print(
    increase_coverage_df[
        "coverage_percent"
    ].describe()
)

print("\nIntervals with 100% coverage:")
print(
    (
        increase_coverage_df[
            "coverage_percent"
        ] == 100
    ).sum()
)

print("\nIntervals with less than 100% coverage:")
print(
    (
        increase_coverage_df[
            "coverage_percent"
        ] < 100
    ).sum()
)

In [ ]:
# ---------------------------------------------------------
# Step 18: Reliable intervals between positive increases
# ---------------------------------------------------------

reliable_increase_intervals = increase_coverage_df[
    increase_coverage_df["coverage_percent"] == 100
].copy()

reliable_waiting_times = (
    reliable_increase_intervals[
        "days_between_increases"
    ]
)

print("Reliable increase intervals:")
print(len(reliable_waiting_times))

print("\nSummary statistics:")
print(reliable_waiting_times.describe())

print("\nMost common interval lengths:")
print(
    reliable_waiting_times
    .value_counts()
    .sort_values(ascending=False)
    .head(15)
)

print("\nPercentiles:")
print(
    reliable_waiting_times.quantile(
        [0.50, 0.75, 0.90, 0.95, 0.99]
    )
)

print("\nIntervals greater than 30 days:")
print(
    (reliable_waiting_times > 30).sum()
)

print("\nMaximum reliable interval:")
print(
    reliable_waiting_times.max()
)

In [ ]:
# ---------------------------------------------------------
# Step 19: Waiting time between increases by fullness state
# ---------------------------------------------------------

positive_state_events = daily_df[
    daily_df["behaviour_type"] == "Increase"
][
    ["serialNumber", "timestamp", "latestFullness"]
].copy()

positive_state_events = positive_state_events.sort_values(
    ["serialNumber", "timestamp"]
)

# State reached at the previous positive increase
positive_state_events["previous_increase_state"] = (
    positive_state_events
    .groupby("serialNumber")["latestFullness"]
    .shift(1)
)

positive_state_events["previous_increase_date"] = (
    positive_state_events
    .groupby("serialNumber")["timestamp"]
    .shift(1)
)

positive_state_events["days_between_increases"] = (
    positive_state_events["timestamp"]
    - positive_state_events["previous_increase_date"]
).dt.days

# Attach the coverage information calculated earlier
waiting_state_df = positive_state_events.merge(
    increase_coverage_df[
        [
            "serialNumber",
            "timestamp",
            "coverage_percent"
        ]
    ],
    on=["serialNumber", "timestamp"],
    how="left"
)

reliable_waiting_state = waiting_state_df[
    waiting_state_df["coverage_percent"] == 100
].copy()

print("Reliable waiting intervals by previous increase state:")

print(
    reliable_waiting_state
    .groupby("previous_increase_state")[
        "days_between_increases"
    ]
    .agg(
        ["count", "mean", "median", "std", "min", "max"]
    )
    .round(2)
)

print("\nMedian waiting time by state:")
print(
    reliable_waiting_state
    .groupby("previous_increase_state")[
        "days_between_increases"
    ]
    .median()
)

In [ ]:
# ---------------------------------------------------------
# Step 20: Check for potential emptying between increases
# ---------------------------------------------------------

potential_events = daily_df[
    daily_df["behaviour_type"] == "Potential Emptying"
][
    ["serialNumber", "timestamp"]
].copy()

def had_emptying_between(row):
    bin_id = row["serialNumber"]
    start = row["previous_increase_date"]
    end = row["timestamp"]

    events = potential_events[
        (potential_events["serialNumber"] == bin_id) &
        (potential_events["timestamp"] > start) &
        (potential_events["timestamp"] < end)
    ]

    return len(events) > 0


reliable_waiting_state[
    "emptying_between"
] = reliable_waiting_state.apply(
    had_emptying_between,
    axis=1
)

print("Potential emptying between consecutive increases:")
print(
    reliable_waiting_state[
        "emptying_between"
    ].value_counts()
)

print("\nPercentage:")
print(
    reliable_waiting_state[
        "emptying_between"
    ]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

print("\nBy previous increase state:")
print(
    pd.crosstab(
        reliable_waiting_state[
            "previous_increase_state"
        ],
        reliable_waiting_state[
            "emptying_between"
        ],
        normalize="index"
    ).mul(100).round(2)
)

In [ ]:
# ---------------------------------------------------------
# Step 21: Accumulation behaviour by bin
# ---------------------------------------------------------

bin_accumulation = (
    accumulation_df
    .groupby("serialNumber")
    .agg(
        accumulation_days=("behaviour_type", "size"),

        increase_days=(
            "behaviour_type",
            lambda x: (x == "Increase").sum()
        ),

        no_change_days=(
            "behaviour_type",
            lambda x: (x == "No Change").sum()
        )
    )
)

bin_accumulation["increase_probability"] = (
    bin_accumulation["increase_days"]
    / bin_accumulation["accumulation_days"]
)

print("Accumulation behaviour by bin:")
print(bin_accumulation.round(4))

print("\nDistribution of bin-level increase probabilities:")
print(
    bin_accumulation[
        "increase_probability"
    ].describe().round(4)
)

print("\nLowest 5:")
print(
    bin_accumulation[
        "increase_probability"
    ].nsmallest(5).round(4)
)

print("\nHighest 5:")
print(
    bin_accumulation[
        "increase_probability"
    ].nlargest(5).round(4)
)

In [ ]:
# ---------------------------------------------------------
# Step 22: Check accumulation behaviour by calendar month
# ---------------------------------------------------------

monthly_accumulation = accumulation_df.copy()

monthly_accumulation["month"] = (
    monthly_accumulation["timestamp"].dt.month
)

monthly_summary = (
    monthly_accumulation
    .groupby("month")
    .agg(
        observations=("behaviour_type", "size"),

        increases=(
            "behaviour_type",
            lambda x: (x == "Increase").sum()
        ),

        no_changes=(
            "behaviour_type",
            lambda x: (x == "No Change").sum()
        )
    )
)

monthly_summary["increase_probability"] = (
    monthly_summary["increases"]
    / monthly_summary["observations"]
)

monthly_summary["increase_probability_pct"] = (
    monthly_summary["increase_probability"] * 100
)

print("Accumulation behaviour by month:")
print(monthly_summary.round(4))

print("\nMonthly increase probability range:")
print(
    "Minimum:",
    round(
        monthly_summary[
            "increase_probability_pct"
        ].min(), 2
    ),
    "%"
)

print(
    "Maximum:",
    round(
        monthly_summary[
            "increase_probability_pct"
        ].max(), 2
    ),
    "%"
)

In [ ]:
# ---------------------------------------------------------
# Step 23: Prepare candidate simulation random variables
# ---------------------------------------------------------

# RV1:
# Probability of No Change vs Increase
# conditional on current fullness state

rv1_table = pd.crosstab(
    accumulation_df["previous_fullness"],
    accumulation_df["behaviour_type"],
    normalize="index"
).mul(100).round(2)

print("RV1 - Daily accumulation occurrence (%)")
print(rv1_table)


# RV2:
# Increase magnitude conditional on an increase occurring
# and conditional on current fullness state

rv2_table = pd.crosstab(
    increase_df["previous_fullness"],
    increase_df["fullness_change"],
    normalize="index"
).mul(100).round(2)

print("\nRV2 - Increase magnitude given an increase (%)")
print(rv2_table)


# Check sample sizes supporting RV1
print("\nRV1 sample size by current fullness:")
print(
    accumulation_df["previous_fullness"]
    .value_counts()
    .sort_index()
)


# Check sample sizes supporting RV2
print("\nRV2 sample size by current fullness:")
print(
    increase_df["previous_fullness"]
    .value_counts()
    .sort_index()
)

In [ ]:
# ---------------------------------------------------------
# Step 24: Validate RV1 + RV2 decomposition
# ---------------------------------------------------------

validation_rows = []

states = sorted(
    accumulation_df["previous_fullness"].unique()
)

for state in states:

    state_data = accumulation_df[
        accumulation_df["previous_fullness"] == state
    ]

    total = len(state_data)

    # Probability of no change
    p_no_change = (
        (state_data["behaviour_type"] == "No Change").mean()
    )

    validation_rows.append(
        {
            "previous_fullness": state,
            "next_fullness": state,
            "reconstructed_probability": p_no_change
        }
    )

    # Probability of increase
    p_increase = (
        (state_data["behaviour_type"] == "Increase").mean()
    )

    state_increases = increase_df[
        increase_df["previous_fullness"] == state
    ]

    if len(state_increases) > 0:

        magnitude_probs = (
            state_increases["fullness_change"]
            .value_counts(normalize=True)
        )

        for magnitude, p_magnitude in magnitude_probs.items():

            next_state = state + magnitude

            validation_rows.append(
                {
                    "previous_fullness": state,
                    "next_fullness": next_state,
                    "reconstructed_probability":
                        p_increase * p_magnitude
                }
            )

validation_df = pd.DataFrame(validation_rows)

validation_table = (
    validation_df
    .pivot_table(
        index="previous_fullness",
        columns="next_fullness",
        values="reconstructed_probability",
        aggfunc="sum",
        fill_value=0
    )
    .mul(100)
    .round(2)
)

print("Reconstructed accumulation transition probabilities (%):")
print(validation_table)

print("\nRow totals:")
print(
    validation_table.sum(axis=1).round(2)
)

In [ ]:
# ---------------------------------------------------------
# Step 25: Export simulation random-variable inputs
# ---------------------------------------------------------

# RV1: Probability of No Change vs Increase
rv1_export = pd.crosstab(
    accumulation_df["previous_fullness"],
    accumulation_df["behaviour_type"],
    normalize="index"
).reset_index()

rv1_export.columns.name = None

# RV2: Increase magnitude conditional on an increase
rv2_export = pd.crosstab(
    increase_df["previous_fullness"],
    increase_df["fullness_change"],
    normalize="index"
).reset_index()

rv2_export.columns.name = None

# Make column names clearer
rv2_export = rv2_export.rename(
    columns={
        2.0: "increase_2",
        4.0: "increase_4",
        6.0: "increase_6",
        8.0: "increase_8",
        10.0: "increase_10"
    }
)

# Export
rv1_export.to_csv(
    "simulation_RV1_accumulation_occurrence.csv",
    index=False
)

rv2_export.to_csv(
    "simulation_RV2_increase_magnitude.csv",
    index=False
)

print("RV1 simulation input:")
print(rv1_export)

print("\nRV2 simulation input:")
print(rv2_export)

print("\nFiles created:")
print("simulation_RV1_accumulation_occurrence.csv")
print("simulation_RV2_increase_magnitude.csv")

In [ ]:
# ---------------------------------------------------------
# Step 26: Final validation of exported simulation inputs
# ---------------------------------------------------------

# Reload exported files
rv1_check = pd.read_csv(
    "simulation_RV1_accumulation_occurrence.csv"
)

rv2_check = pd.read_csv(
    "simulation_RV2_increase_magnitude.csv"
)

# Probability columns
rv1_probability_cols = [
    "Increase",
    "No Change"
]

rv2_probability_cols = [
    "increase_2",
    "increase_4",
    "increase_6",
    "increase_8",
    "increase_10"
]

# Calculate row totals
rv1_check["probability_sum"] = (
    rv1_check[rv1_probability_cols].sum(axis=1)
)

rv2_check["probability_sum"] = (
    rv2_check[rv2_probability_cols].sum(axis=1)
)

print("RV1 validation:")
print(rv1_check)

print("\nRV1 missing values:")
print(rv1_check.isna().sum().sum())

print("\nRV1 probability sums:")
print(rv1_check["probability_sum"].round(6))


print("\n-----------------------------------")


print("\nRV2 validation:")
print(rv2_check)

print("\nRV2 missing values:")
print(rv2_check.isna().sum().sum())

print("\nRV2 probability sums:")
print(rv2_check["probability_sum"].round(6))